# NYC Taxi DuckDB SQL Agent Homework Notebook

Run the cells from top to bottom. The first code cell deletes `taxi.db` if it already exists so the database starts fresh.

In [56]:
# Optional: install dependencies if needed
# Run this once if your notebook environment does not already have these packages.

# !uv add duckdb pydantic-ai

import os
import urllib.request
from typing import Any
from pathlib import Path

import duckdb


In [1]:
# Start clean: delete taxi.db if it exists

DB_FILE = "taxi.db"

if Path(DB_FILE).exists():
    Path(DB_FILE).unlink()
    print(f"Deleted existing {DB_FILE}")
else:
    print(f"No existing {DB_FILE} found")

Deleted existing taxi.db


## 1. Build the `sql_tools.py` logic inside the notebook

This cell contains the same responsibilities as `sql_tools.py`:

- download the NYC taxi parquet file
- create/load the DuckDB database
- define `SQLTools.get_schema()`
- define `SQLTools.run_sql()`

In [28]:

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
PARQUET_FILE = "yellow_tripdata_2024-01.parquet"
DB_FILE = "taxi.db"


def setup_database() -> int:
    """Download the parquet file and load it into DuckDB."""
    if not os.path.exists(PARQUET_FILE):
        print(f"Downloading {DATA_URL}...")
        urllib.request.urlretrieve(DATA_URL, PARQUET_FILE)

    with duckdb.connect(DB_FILE) as con:
        con.execute(
            f"""
            CREATE TABLE IF NOT EXISTS trips AS
            SELECT * FROM '{PARQUET_FILE}'
            """
        )

        count = con.execute("SELECT COUNT(*) FROM trips").fetchone()[0]

    print(f"Loaded {count:,} rows")
    return count


class SQLTools:
    """Tools the agent can use to inspect and query the taxi database."""

    def __init__(self, db_path: str = DB_FILE) -> None:
        self.db_path = db_path

    def _connect(self) -> duckdb.DuckDBPyConnection:
        return duckdb.connect(self.db_path)

    def get_schema(self) -> str:
        """Return the trips table schema with column names and types."""
        with self._connect() as conn:
            rows = conn.execute("DESCRIBE trips").fetchall()

        lines = ["column_name | column_type"]
        lines.extend(f"{row[0]} | {row[1]}" for row in rows)
        return " ".join(lines)

    def run_sql(self, query: str) -> str:
        """Execute SQL and return column headers plus up to 50 rows as text."""
        safe_query = query.strip().rstrip(";")
        limited_query = f"SELECT * FROM ({safe_query}) AS agent_query LIMIT 50"

        with self._connect() as conn:
            result = conn.execute(limited_query)
            columns = [desc[0] for desc in result.description]
            rows: list[tuple[Any, ...]] = result.fetchall()

        output = [" | ".join(columns)]
        output.extend(" | ".join(str(value) for value in row) for row in rows)
        return " ".join(output)


## 2. Create the database

In [3]:
count = setup_database()
count

Loaded 2,964,624 rows


2964624

## 3. Query the database directly

Before using the AI agent, it helps to verify answers with plain SQL.

In [4]:
con = duckdb.connect("taxi.db")

In [5]:
con.execute("SELECT COUNT(*) AS total_rows FROM trips").fetchdf()

,total_rows
0,2964624


In [6]:
con.execute(""" SELECT COUNT(*) AS trips_more_than_5_passengers FROM trips WHERE passenger_count > 5 """).fetchdf()

,trips_more_than_5_passengers
0,22413


In [7]:
con.execute("""SELECT AVG(trip_distance) AS avg_trip_distance FROM trips WHERE passenger_count = 2""").fetchdf()

,avg_trip_distance
0,3.782764


In [8]:
con.close()

## 4. Build the `sql_agent.py` logic inside the notebook

This cell contains the same responsibilities as `sql_agent.py`:

- define the `SQLResult` output model
- create the PydanticAI agent
- define `ask_agent()`

In [9]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent


class SQLResult(BaseModel):
    """Structured output returned by the SQL agent."""

    sql_query: str = Field(description="The SQL query used to answer the question")
    result_text: str = Field(description="A clear text summary of the SQL result")
    row_count: int = Field(description="The number of result rows returned or summarized")


sql_tools = SQLTools()

agent = Agent(
    "openai:gpt-4o-mini",
    output_type=SQLResult,
    tools=[sql_tools.get_schema, sql_tools.run_sql],
    instructions="""
You are a careful SQL assistant that answers questions about the DuckDB table named trips.

Rules:
1. Always call get_schema first before writing SQL.
2. After reading the schema, write a DuckDB-compatible SQL query.
3. Call run_sql with the query.
4. Return the exact SQL query you used in sql_query.
5. Return a concise answer in result_text, including the important numeric result.
6. Return row_count as the number of rows in the final result, usually 1 for aggregate questions.
""".strip(),
)


async def ask_agent(question: str) -> SQLResult:
    """Ask the agent a question and return its structured output."""
    result = await agent.run(question)
    return result.output


## 5. Ask the agent homework questions

In [10]:
answer = await ask_agent("What's the average trip distance for rides with 2 passengers?")
answer

SQLResult(sql_query='SELECT AVG(trip_distance) AS average_trip_distance FROM trips WHERE passenger_count = 2;', result_text='The average trip distance for rides with 2 passengers is approximately 3.78 miles.', row_count=1)

In [11]:
print("SQL query:")
print(answer.sql_query)
print("Result:")
print(answer.result_text)

SQL query:
SELECT AVG(trip_distance) AS average_trip_distance FROM trips WHERE passenger_count = 2;
Result:
The average trip distance for rides with 2 passengers is approximately 3.78 miles.


In [12]:
answer = await ask_agent("How many trips had more than 5 passengers?")
print(answer.model_dump_json(indent=2))

{
  "sql_query": "SELECT COUNT(*) AS num_trips FROM trips WHERE passenger_count > 5;",
  "result_text": "There were 22,413 trips with more than 5 passengers.",
  "row_count": 1
}


In [13]:
answer = await ask_agent("What is the most common payment type?")
print(answer.model_dump_json(indent=2))

{
  "sql_query": "SELECT payment_type, COUNT(*) as count FROM trips GROUP BY payment_type ORDER BY count DESC LIMIT 1;",
  "result_text": "The most common payment type is 1 with 2,319,046 occurrences.",
  "row_count": 1
}


In [14]:
answer = await ask_agent("Which hour of the day has the highest average fare amount?")
print(answer.model_dump_json(indent=2))

{
  "sql_query": "SELECT EXTRACT(HOUR FROM tpep_pickup_datetime) AS hour, AVG(fare_amount) AS average_fare\nFROM trips\nGROUP BY hour\nORDER BY average_fare DESC\nLIMIT 1;",
  "result_text": "The hour of the day with the highest average fare amount is 5, with an average fare of approximately $26.62.",
  "row_count": 1
}


## 6. Extra direct SQL queries for Question 6

In [15]:
queries = {
    "avg_tip_credit_card": """
        SELECT AVG(tip_amount) AS avg_tip_amount
        FROM trips
        WHERE payment_type = 1
    """,
    "top_pickup_location": """
        SELECT PULocationID, COUNT(*) AS trip_count
        FROM trips
        GROUP BY PULocationID
        ORDER BY trip_count DESC
        LIMIT 1
    """,
    "avg_fare_long_trips": """
        SELECT AVG(fare_amount) AS avg_fare_amount
        FROM trips
        WHERE trip_distance > 10
    """,
    "zero_passenger_trips": """
        SELECT COUNT(*) AS zero_passenger_trips
        FROM trips
        WHERE passenger_count = 0
    """,
    "busiest_day_of_week": """
        SELECT
            strftime(tpep_pickup_datetime, '%A') AS day_of_week,
            COUNT(*) AS trip_count
        FROM trips
        GROUP BY day_of_week
        ORDER BY trip_count DESC
        LIMIT 1
    """,
}

with duckdb.connect(DB_FILE) as con:
    for name, query in queries.items():
        print(f"{name}")
        display(con.execute(query).fetchdf())

avg_tip_credit_card


,avg_tip_amount
0,4.169671


top_pickup_location


,PULocationID,trip_count
0,132,145240


avg_fare_long_trips


,avg_fare_amount
0,62.880812


zero_passenger_trips


,zero_passenger_trips
0,31465


busiest_day_of_week


,day_of_week,trip_count
0,Wednesday,495032


## 7. Ask the agent the extra Question 6 prompts

In [16]:
questions = [
    "What is the average tip amount for credit card payments?",
    "Which pickup location (PULocationID) has the most trips?",
    "What is the average fare for trips longer than 10 miles?",
    "How many trips had zero passengers recorded?",
    "What is the busiest day of the week for taxi trips?",
]

for question in questions:
    print("=" * 80)
    print(question)
    answer = await ask_agent(question)
    print("SQL:")
    print(answer.sql_query)
    print("Result:")
    print(answer.result_text)
    print()

What is the average tip amount for credit card payments?
SQL:
SELECT AVG(tip_amount) AS average_tip
FROM trips
WHERE payment_type = 1;
Result:
The average tip amount for credit card payments is approximately 4.17.

Which pickup location (PULocationID) has the most trips?
SQL:
SELECT PULocationID, COUNT(*) AS trip_count
FROM trips
GROUP BY PULocationID
ORDER BY trip_count DESC
LIMIT 1;
Result:
The pickup location with the most trips is PULocationID 132, totaling 145240 trips.

What is the average fare for trips longer than 10 miles?
SQL:
SELECT AVG(fare_amount) AS average_fare FROM trips WHERE trip_distance > 10;
Result:
The average fare for trips longer than 10 miles is approximately $62.88.

How many trips had zero passengers recorded?
SQL:
SELECT COUNT(*) as zero_passenger_trips FROM trips WHERE passenger_count = 0;
Result:
There were 31,465 trips recorded with zero passengers.

What is the busiest day of the week for taxi trips?
SQL:
SELECT strftime('%w', tpep_pickup_datetime) AS da

## 8. Set up for tests

In [22]:
from judge import assert_criteria
from sql_agent import agent
from utils import collect_tools

In [23]:
def scalar(sql: str):
    """Run a direct DuckDB query to get a reliable expected answer."""
    with duckdb.connect("taxi.db") as conn:
        return conn.execute(sql).fetchone()[0]

In [24]:
async def run_notebook_test(test_func):
    """Run one async test and print notebook-friendly output."""
    try:
        await test_func()
        print(f"✅ PASSED: {test_func.__name__}")
    except AssertionError as e:
        print(f"❌ FAILED: {test_func.__name__}")
        print(e)
    except Exception as e:
        print(f"💥 ERROR: {test_func.__name__}")
        print(type(e).__name__, e)
        traceback.print_exc()

In [25]:
async def test_agent_counts_trips_with_more_than_five_passengers():
    expected = scalar("SELECT COUNT(*) FROM trips WHERE passenger_count > 5")

    result = await agent.run("How many trips had more than 5 passengers?")
    output = result.output

    print("Question: How many trips had more than 5 passengers?")
    print("SQL query:", output.sql_query)
    print("Agent answer:", output.result_text)
    print("Expected:", expected)

    assert isinstance(output.sql_query, str)
    assert output.sql_query.strip() != ""
    assert str(expected) in output.result_text.replace(",", "")


async def test_agent_gets_schema_before_running_sql():
    result = await agent.run("What is the most common payment type?")
    tool_names = collect_tools(result.all_messages())

    print("Question: What is the most common payment type?")
    print("Tools used:", tool_names)

    assert tool_names[0] == "get_schema"
    assert "run_sql" in tool_names


async def test_llm_judge_highest_average_fare_hour():
    question = "Which hour of the day has the highest average fare amount?"
    result = await agent.run(question)

    print("Question:", question)
    print("SQL query:", result.output.sql_query)
    print("Agent answer:", result.output.result_text)

    await assert_criteria(
        question=question,
        answer=result.output,
        criteria=[
            "the SQL query correctly calculates average fare by hour of day",
            "the result identifies a specific hour as having the highest average fare",
            "the result includes the actual average fare amount",
        ],
    )


async def test_average_tip_for_credit_card_payments():
    expected = scalar("SELECT AVG(tip_amount) FROM trips WHERE payment_type = 1")

    result = await agent.run("What is the average tip amount for credit card payments?")
    output = result.output
    tool_names = collect_tools(result.all_messages())

    print("Question: What is the average tip amount for credit card payments?")
    print("Tools used:", tool_names)
    print("SQL query:", output.sql_query)
    print("Agent answer:", output.result_text)
    print("Expected:", expected)

    assert tool_names[0] == "get_schema"
    assert "run_sql" in tool_names
    assert "tip_amount" in output.sql_query
    assert "payment_type" in output.sql_query
    assert f"{expected:.2f}" in output.result_text or str(round(expected, 2)) in output.result_text


async def test_pickup_location_with_most_trips():
    expected = scalar(
        """
        SELECT PULocationID
        FROM trips
        GROUP BY PULocationID
        ORDER BY COUNT(*) DESC
        LIMIT 1
        """
    )

    result = await agent.run("Which pickup location (PULocationID) has the most trips?")
    output = result.output

    print("Question: Which pickup location has the most trips?")
    print("SQL query:", output.sql_query)
    print("Agent answer:", output.result_text)
    print("Expected:", expected)

    assert "PULocationID" in output.sql_query or "pulocationid" in output.sql_query.lower()
    assert "COUNT" in output.sql_query.upper()
    assert str(expected) in output.result_text


async def test_average_fare_for_trips_longer_than_ten_miles():
    expected = scalar("SELECT AVG(fare_amount) FROM trips WHERE trip_distance > 10")

    result = await agent.run("What is the average fare for trips longer than 10 miles?")
    output = result.output

    print("Question: What is the average fare for trips longer than 10 miles?")
    print("SQL query:", output.sql_query)
    print("Agent answer:", output.result_text)
    print("Expected:", expected)

    assert "fare_amount" in output.sql_query
    assert "trip_distance" in output.sql_query
    assert f"{expected:.2f}" in output.result_text or str(round(expected, 2)) in output.result_text


async def test_zero_passenger_trips_filters_on_passenger_count():
    expected = scalar("SELECT COUNT(*) FROM trips WHERE passenger_count = 0")

    result = await agent.run("How many trips had zero passengers recorded?")
    output = result.output

    print("Question: How many trips had zero passengers recorded?")
    print("SQL query:", output.sql_query)
    print("Agent answer:", output.result_text)
    print("Expected:", expected)

    assert "passenger_count" in output.sql_query
    assert str(expected) in output.result_text.replace(",", "")


async def test_busiest_day_of_week_for_taxi_trips():
    expected = scalar(
        """
        SELECT strftime(tpep_pickup_datetime, '%A') AS day_name
        FROM trips
        GROUP BY day_name
        ORDER BY COUNT(*) DESC
        LIMIT 1
        """
    )

    result = await agent.run("What is the busiest day of the week for taxi trips?")
    output = result.output

    print("Question: What is the busiest day of the week for taxi trips?")
    print("SQL query:", output.sql_query)
    print("Agent answer:", output.result_text)
    print("Expected:", expected)

    assert "tpep_pickup_datetime" in output.sql_query
    assert "COUNT" in output.sql_query.upper()
    assert str(expected) in output.result_text

## 9. All all the tests

In [26]:
tests = [
    test_agent_counts_trips_with_more_than_five_passengers,
    test_agent_gets_schema_before_running_sql,
    test_llm_judge_highest_average_fare_hour,
    test_average_tip_for_credit_card_payments,
    test_pickup_location_with_most_trips,
    test_average_fare_for_trips_longer_than_ten_miles,
    test_zero_passenger_trips_filters_on_passenger_count,
    test_busiest_day_of_week_for_taxi_trips,
]

for test in tests:
    await run_notebook_test(test)
    print("-" * 80)

Question: How many trips had more than 5 passengers?
SQL query: SELECT COUNT(*) AS trip_count FROM trips WHERE passenger_count > 5;
Agent answer: There were 22,413 trips with more than 5 passengers.
Expected: 22413
✅ PASSED: test_agent_counts_trips_with_more_than_five_passengers
--------------------------------------------------------------------------------
Question: What is the most common payment type?
Tools used: ['get_schema', 'get_schema', 'run_sql', 'run_sql', 'final_result', 'final_result']
✅ PASSED: test_agent_gets_schema_before_running_sql
--------------------------------------------------------------------------------
Question: Which hour of the day has the highest average fare amount?
SQL query: SELECT EXTRACT(HOUR FROM tpep_pickup_datetime) AS hour, AVG(fare_amount) AS average_fare
FROM trips
GROUP BY hour
ORDER BY average_fare DESC
LIMIT 1;
Agent answer: The hour with the highest average fare amount is 5, with an average fare of approximately $26.62.
✅ PASSED: test_llm_ju

In [27]:
con.close()